# 01 · Agents and traces: one question, three runtimes

The improvement loop never talks to an agent framework directly. It only sees **MLflow traces** and an
**`AgentSpec`** (an `AGENTS.md` version plus skill versions). That's why OpenClaw and Hermes Agent are
interchangeable here: each runtime just has to produce the same trace shape.

```
acme_support_agent (AGENT)          tags: topic, backend, prompt_version, skills, source
  └─ openclaw.agent | hermes.agent  (CHAIN)
       ├─ llm_call (LLM)             each model turn
       ├─ read (TOOL)                each file the agent opened, with its path
       └─ llm_call (LLM)
```

In [1]:
from sia import notebook as nb

nb.setup(backend="offline")
nb.available_backends()

tracking_uri  sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/mlflow.db
experiment    acme-support-notebooks (id 1)
backend       offline
agent_model   anthropic:/claude-haiku-5-5
judge_model   anthropic:/claude-opus-5-5
llm_judges    True


{'offline': 'ok', 'direct': 'ok', 'openclaw': 'ok', 'hermes': 'ok'}

## Render an agent version into a workspace

Before every answer, the harness renders the requested version into a fresh directory:
* `AGENTS.md` from the Prompt Registry;
* `knowledge/` (the policies);
* `skills/`, if the spec has any.

We'll register the hand-written Level 2 fix as v2 (not promoted), because it actually reads the knowledge
files. That makes the tool spans interesting.

In [2]:
import mlflow
from sia import config
from sia.agent import AgentSpec, pin, resolve, workspace_for, register_candidate
from sia.scenario import SCENARIO_DIR

nb.ensure_production()
fix = (SCENARIO_DIR / "fixes" / "level2_manual_AGENTS.md").read_text()
versions = [int(v.version) for v in mlflow.MlflowClient().search_prompt_versions(config.PROMPT_NAME)]
V2 = next((v for v in versions if mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}/{v}").template == fix), None) \
     or register_candidate(fix, "Level 2 manual fix", author="notebook", level=2)

agents_md, skills = resolve(V2)
ws = workspace_for(agents_md, skills)
print("spec:", pin(V2).label, "->", ws.name)
print(sorted(str(p.relative_to(ws)) for p in ws.rglob("*") if p.is_file()))

spec: v2 -> v2-dcaef34d
['AGENTS.md', 'knowledge/account_access.md', 'knowledge/billing.md', 'knowledge/product_howto.md', 'knowledge/support_process.md']


## Ask the same question on each runtime

`run_agent()` is the single traced entry point; `backend=` picks the runtime per call.
* **Offline:** returns instantly.
* **OpenClaw and Hermes:** a few seconds each on Claude Haiku 5.5, or 1–3 minutes each on a local Ollama model.

They run only if `available_backends()` says `ok`.

In [3]:
import time
from sia.agent import run_agent

QUESTION = "Can we set up SSO with Okta on our Pro plan?"
results = {}

def ask(backend):
    t = time.perf_counter()                       # monotonic: not fooled if the laptop sleeps mid-answer
    answer = run_agent(QUESTION, prompt_ref=V2, backend=backend)
    trace = nb.last_trace()
    results[backend] = {"seconds": round(time.perf_counter() - t, 1), "answer": answer, "trace": trace}
    print(f"[{backend}] {results[backend]['seconds']}s\n{answer[:300]}\n")
    nb.show(trace)

ask("offline")

[offline] 1.5s
No. SSO is available on the Enterprise plan only.

AGENT      acme_support_agent      0.1s  'No. SSO is available on the Enterprise plan only.'
  └─ TOOL       read                    0.0s  knowledge/account_access.md
tags: {'topic': 'account_access', 'backend': 'offline', 'prompt_version': '2', 'skills': '', 'source': 'eval'}


### OpenClaw

OpenClaw runs as a subprocess (`openclaw agent --local --json`) against an isolated profile. Its `--json` output
only summarizes tool use, so after each answer the harness exports OpenClaw's own session trajectory and replays
it as LLM and TOOL spans with the original timestamps (`src/sia/openclaw_trace.py`).

In [4]:
avail = nb.available_backends()
if avail["openclaw"] == "ok":
    from sia import openclaw_setup
    openclaw_setup.write_config()          # isolated profile under notebooks/.mlflow/openclaw
    ask("openclaw")
else:
    print("skipping OpenClaw:", avail["openclaw"])

[openclaw] 13.7s
No, SSO with Okta isn't available on the Pro plan. Acme Cloud's SSO (SAML/OIDC) is available on the **Enterprise plan only**, so Okta SSO would require upgrading to Enterprise.

If you're interested in Enterprise, I can escalate this to a human teammate who can talk through pricing and setup. Pro in

AGENT      acme_support_agent     12.6s  "No, SSO with Okta isn't available on the Pro plan. Acme Cloud's SSO (S"
  └─ CHAIN      openclaw.agent         12.6s  
    └─ LLM        llm_call                1.4s  -> tools [{'name': 'read', 'arguments': {'path': 'knowledge/billing.md', 'optional': True}}, {'name': 'read', 'arguments': {'path': 'knowledge/account_access.md', 'optional': True}}, {'name': 'read', 'arguments': {'path': 'knowledge/product_howto.md', 'optional': True}}]
    └─ TOOL       read                    0.0s  knowledge/product_howto.md
    └─ TOOL       read                    0.0s  knowledge/account_access.md
    └─ TOOL       read                    0.0s  k

### Hermes Agent

Hermes runs in-process through its Python API (`run_agent.AIAgent`), with its own home directory per agent
version. Two small hooks make its work visible:
* a subclass wraps its model calls in LLM spans;
* its `tool_start` / `tool_complete` callbacks become TOOL spans (`src/sia/hermes_runtime.py`).

In [5]:
if avail["hermes"] == "ok":
    ask("hermes")
else:
    print("skipping Hermes:", avail["hermes"])

🧾 Request debug dump written to: /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/hermes-mlflow/homes/v2-dcaef34d/sessions/request_dump_sia-d47b072bf48c_20261019_235456_090346.json
⚠️  API call failed (attempt 1/3): BadRequestError [HTTP 400]
   🔌 Provider: anthropic  Model: claude-haiku-5-5
   🌐 Endpoint: 
   📝 Error: HTTP 400: Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.
   📋 Details: {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CgCpAa8urVdkUVv5cwF6v'}
   ⏱️  Elapsed: 0.78s  Context: 2 msgs, ~2,068 tokens
❌ Non-retryable error (HTTP 400): HTTP 400: Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.
❌ Non-retryable client error (HTTP 400)

## Same shape, comparable signal

Because every runtime leaves TOOL spans with file paths, one deterministic scorer can judge *how* each answer
was found. For an SSO question, the right file is `knowledge/account_access.md`.

In [ ]:
import pandas as pd
from sia import signals

pd.DataFrame([{
    "backend": b, "seconds": r["seconds"],
    "files read": ", ".join(dict.fromkeys(signals.knowledge_reads(r["trace"]))) or "-",
    "read_right_policy": signals.read_right_policy(trace=r["trace"]).value,
    "focused_reading": signals.focused_reading(trace=r["trace"]).value,
    "answer": r["answer"][:90],
} for b, r in results.items()])

## How the spans get there

The two runtimes need different techniques, but end up with the same trace shape. Here are the key lines:

In [ ]:
import inspect
from sia import openclaw_trace, hermes_runtime

print(inspect.getsource(openclaw_trace.replay).split("\n    for e in events")[0][:900])
print("...")
print(inspect.getsource(hermes_runtime.traced_agent_class)[:1100])

## Finding traces later

Every answer is tagged, so search is just a filter string. Production traffic is tagged `source=production`, and
evaluation runs are tagged `source=eval`.

In [ ]:
from sia import tracking

traces = tracking.search("trace.name = 'acme_support_agent'")
pd.DataFrame([{"trace_id": t.info.trace_id, **{k: t.info.tags.get(k) for k in ("backend", "prompt_version", "source", "topic")}}
              for t in traces]).head(10)

**Next:** 02 · Signal. Who says an answer is good or bad, and how much should we trust them?